# A small Transformer encoder on CPU

Combine an input projection, positional encoding, self-attention, and mean pooling to classify short numeric sequences. This is a randomly initialized, one-layer educational encoder, not a pretrained language model.

**Prerequisites:** the NumPy attention lesson and the PyTorch training lesson. **Environment:** base requirements plus CPU PyTorch. **Execution:** restart and run all cells. No pretrained weights, dataset download, or GPU is required.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


## Generate independent sequences

Each sequence belongs to one of two trend classes. Split along the sample axis before training. Real repeated subjects, overlapping time windows, or future forecasting tasks require different split rules.


In [ ]:
rng = np.random.default_rng(SEED)
samples, steps = 400, 24
y = np.repeat([0, 1], samples // 2)
time = np.linspace(-1, 1, steps, dtype=np.float32)
direction = (2 * y - 1).astype(np.float32)
amplitude = rng.uniform(0.4, 1.2, size=samples).astype(np.float32)
offset = rng.uniform(-0.3, 0.3, size=samples).astype(np.float32)
X = (offset[:, None] + direction[:, None] * amplitude[:, None] * time
     + rng.normal(0, 0.15, (samples, steps))).astype(np.float32)[..., None]
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, stratify=y_development, random_state=SEED
)
assert X_train.shape[1:] == (steps, 1)
print('Shape: (samples, time steps, features) =', X.shape)
fig, ax = plt.subplots(figsize=(6, 3))
for label, index in [(0, 0), (1, samples // 2)]:
    ax.plot(time, X[index, :, 0], label=['decreasing', 'increasing'][label])
ax.set(xlabel='Time', ylabel='Signal', title='Independent synthetic sequences')
ax.legend()
plt.show()


## Add position before the encoder

Self-attention processes interactions among positions. Sinusoidal encodings tell the model where each value occurs. The encoder below uses two attention heads with model width 16. Mean pooling produces one vector per sequence. All sequences have equal length, so this example needs no padding mask. We use unrestricted encoder attention because the entire input sequence is available for classification.


In [ ]:
class TrendTransformer(nn.Module):
    def __init__(self, steps=24, width=16):
        super().__init__()
        self.project = nn.Linear(1, width)
        positions = torch.arange(steps).float().unsqueeze(1)
        frequencies = torch.exp(torch.arange(0, width, 2).float() * (-np.log(10000.0) / width))
        encoding = torch.zeros(steps, width)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer('positions', encoding.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d_model=width, nhead=2, dim_feedforward=32,
                                           dropout=0.0, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1, enable_nested_tensor=False)
        self.classifier = nn.Linear(width, 2)
    def forward(self, sequences):
        projected = self.project(sequences) + self.positions[:, :sequences.shape[1]]
        encoded = self.encoder(projected)
        return self.classifier(encoded.mean(dim=1))

torch.manual_seed(SEED)
model = TrendTransformer(steps=steps).to(DEVICE)
assert model(torch.tensor(X_train[:4])).shape == (4, 2)
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))


## Train on training data and select on validation data

Each batch clears accumulated gradients, computes the loss, backpropagates, and updates parameters. `train()` enables training behavior; `eval()` and `no_grad()` are used for evaluation. The helper below retains a **copy** of the weights with the lowest validation loss. The test set is not used to choose an epoch or model.

Seeds make this small CPU experiment repeatable in the same environment. Bitwise equality across different library versions or hardware is not promised.


In [ ]:
def loader(features, targets, shuffle=False, batch_size=64):
    dataset = TensorDataset(torch.as_tensor(features, dtype=torch.float32),
                            torch.as_tensor(targets, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def evaluate_classifier(model, batches):
    model.eval()  # disables dropout and uses inference behavior
    total_loss, total, correct = 0.0, 0, 0
    all_predictions = []
    with torch.no_grad():  # validation does not build a gradient graph
        for features, targets in batches:
            logits = model(features.to(DEVICE))
            targets = targets.to(DEVICE)
            total_loss += nn.functional.cross_entropy(logits, targets, reduction='sum').item()
            prediction = logits.argmax(dim=1)
            correct += (prediction == targets).sum().item()
            total += targets.numel()
            all_predictions.append(prediction.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_predictions)

def fit_classifier(model, train_batches, validation_batches, epochs=10, learning_rate=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_validation_loss = [], None, float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0
        for features, targets in train_batches:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(features), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total += targets.numel()
        validation_loss, validation_accuracy, _ = evaluate_classifier(model, validation_batches)
        history.append({'epoch': epoch, 'train_loss': total_loss / total,
                        'validation_loss': validation_loss,
                        'validation_accuracy': validation_accuracy})
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history)


In [ ]:
history = fit_classifier(model, loader(X_train, y_train, shuffle=True),
                         loader(X_validation, y_validation), epochs=14, learning_rate=0.006)
display(history.tail())
history.plot(x='epoch', y=['train_loss', 'validation_loss'], figsize=(6, 3))
plt.ylabel('Cross-entropy')
plt.show()


## Test the selected epoch

Results describe one generated task and one split. They cannot establish that attention outperforms recurrent networks or that this encoder handles natural language.


In [ ]:
test_loss, test_accuracy, predictions = evaluate_classifier(model, loader(X_test, y_test))
print('Held-out loss:', round(test_loss, 4), 'Accuracy:', round(test_accuracy, 4))
print(classification_report(y_test, predictions, target_names=['decreasing', 'increasing'], zero_division=0))


## Practice

Train a separate model with zero positional encodings. Compare validation results and reason about what information mean pooling loses when positions are absent. Longer or padded sequences need an explicit mask and more memory.

**References:** [TransformerEncoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html), [TransformerEncoder](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html), [Attention Is All You Need](https://arxiv.org/abs/1706.03762).
